### Phase II Work Report: Protocol Metadata and Asset Pricing

Raw blockchain data contains no concept of “US Dollars” or token names; it only recognizes hexadecimal contract addresses and raw integer amounts. The objective of Phase II was to map the topological metadata of the decentralized exchange and establish a standardized fiat pricing vector. This ensures all econometric regressions are based on accurate monetary values, rather than arbitrary token counts.

**Methodology:**

1. **Metadata Association:**  
   I extracted the definitive state configurations for every active liquidity pool. This involved mapping the pool’s contract address to its underlying asset pairs (Token 0 and Token 1), identifying the specific fee tier (e.g., 0.05%, 0.3%), and extracting the immutable `tickSpacing` boundaries.

2. **Decimal Normalization:**  
   ERC-20 tokens utilize disparate decimal configurations (e.g., USDC uses 6 decimals, while WETH uses 18). I cross-referenced token registries to apply the correct fractional divisor to the raw integer amounts from Phase I, converting them into true mathematical unit quantities.

3. **Fiat Pricing Architecture:**  
   To calculate true economic volume and Maximum Extractable Value (MEV) damage, I integrated historical Oracle pricing vectors. By mapping the execution timestamps to the closest available USDC/WETH reference pool data, the system successfully derived the exact US Dollar value of every token at the exact millisecond of the trade.



In [8]:
from config import OUT
import polars as pl
import time
import random
from web3 import Web3, HTTPProvider
from web3.exceptions import Web3RPCError
from eth_abi import decode
from concurrent.futures import ThreadPoolExecutor
import requests
from datetime import datetime, timezone
import io
import zlib
import zipfile
import http.client
import urllib.error
import urllib.request
from datetime import datetime
from dateutil.relativedelta import relativedelta
from pathlib import Path

print(f"Saving data to: {OUT}")

Saving data to: C:\Users\Pouyan\python\thesis\Proposal\FINAL\Thesis_Output


---

## Extracts unique tokens to fetch

this cell loads the previously cleaned pool registry and extracts every unique token address. it prepares a list of tokens that need metadata fetching.

In [10]:
# Load the canonical pool registry
pools = pl.read_parquet("uniswap_v3_pools_clean.parquet")

# Extract every unique token address from both token0 and token1 columns
tokens = pl.concat([
    pools.select(pl.col("token0").alias("token")),
    pools.select(pl.col("token1").alias("token"))
]).unique(subset=["token"]).with_columns(
    pl.col("token").str.to_lowercase()
)

print(f"Total unique tokens canonical Uniswap V3: {tokens.height:,}")
tokens.write_parquet("unique_tokens_to_fetch.parquet")

Total unique tokens canonical Uniswap V3: 56,599


---

## Token Metadata Fetcher (Infura + Multicall3)

this cell connects to the blockchain using a remote provider. it uses a multicall contract to fetch the decimals and symbols for all tokens in batches. it includes retry logic to handle network drops.

In [12]:
# Setup Infura

RPC_URL = "InfuraRPC" #put your own RPC endpoint here

w3 = Web3(
    HTTPProvider(
        RPC_URL,
        request_kwargs={"timeout": 120}
    )
)

if not w3.is_connected():
    raise ConnectionError("Could not connect to Infura.")

MULTICALL3 = "0xcA11bde05977b3631167028862bE2a173976CA11"

DECIMALS_SIG = w3.keccak(text="decimals()")[:4]
SYMBOL_SIG   = w3.keccak(text="symbol()")[:4]

MC3_ABI = [{
    "inputs": [{
        "components": [
            {"name": "target", "type": "address"},
            {"name": "allowFailure", "type": "bool"},
            {"name": "callData", "type": "bytes"}
        ],
        "name": "calls",
        "type": "tuple[]"
    }],
    "name": "aggregate3",
    "outputs": [{
        "components": [
            {"name": "success", "type": "bool"},
            {"name": "returnData", "type": "bytes"}
        ],
        "name": "returnData",
        "type": "tuple[]"
    }],
    "stateMutability": "view",
    "type": "function"
}]

mc3 = w3.eth.contract(
    address=w3.to_checksum_address(MULTICALL3),
    abi=MC3_ABI
)


# Safe Decoders
def safe_decode_decimals(ret):
    if not ret:
        return None
    try:
        return decode(["uint8"], ret)[0]
    except Exception:
        return None


def safe_decode_symbol(ret):
    if not ret:
        return None

    try:
        return decode(["string"], ret)[0]
    except Exception:
        try:
            b = decode(["bytes32"], ret)[0]
            return b.decode("utf-8", "ignore").rstrip("\x00")
        except Exception:
            return None


# Batch Fetching with Retry
def fetch_batch(tokens):

    calls = []

    for t in tokens:
        addr = w3.to_checksum_address(t)
        calls.append((addr, True, DECIMALS_SIG))
        calls.append((addr, True, SYMBOL_SIG))

    delay = 1

    for attempt in range(6):

        try:
            results = mc3.functions.aggregate3(calls).call()
            break

        except Exception as e:

            print(
                f"Retry {attempt+1}/6 "
                f"({len(tokens)} tokens): {type(e).__name__}"
            )

            if attempt == 5:
                print("Batch permanently failed.")
                return []

            time.sleep(delay + random.uniform(0, 0.5))
            delay *= 2

    data = []

    for i, token in enumerate(tokens):

        dec_success, dec_data = results[i * 2]
        sym_success, sym_data = results[i * 2 + 1]

        decimals = safe_decode_decimals(dec_data) if dec_success else None
        symbol = safe_decode_symbol(sym_data) if sym_success else None

        data.append({
            "token": token,
            "decimals": decimals,
            "symbol": symbol,
        })

    return data

# load tokens and prepare batches

tokens_df = pl.read_parquet("unique_tokens_to_fetch.parquet")
tokens = tokens_df["token"].to_list()

# Much friendlier to Infura
BATCH_SIZE = 100

batches = [
    tokens[i:i+BATCH_SIZE]
    for i in range(0, len(tokens), BATCH_SIZE)
]

print(
    f"Fetching metadata for {len(tokens):,} tokens "
    f"in {len(batches):,} batches..."
)


# Execute

all_data = []

with ThreadPoolExecutor(max_workers=2) as executor:

    for i, result in enumerate(executor.map(fetch_batch, batches)):

        all_data.extend(result)

        # Small pause to avoid hammering Infura
        time.sleep(0.2)

        if (i + 1) % 20 == 0:
            processed = min((i + 1) * BATCH_SIZE, len(tokens))
            print(f"Processed {processed:,}/{len(tokens):,}")

# 6. Save

metadata_df = pl.DataFrame(all_data)

metadata_df.write_parquet("token_metadata.parquet")

print("\nDone!")

print(metadata_df.head())

success_count = metadata_df.filter(
    pl.col("decimals").is_not_null()
).height

print(
    f"\nSuccessfully fetched decimals for "
    f"{success_count:,}/{len(tokens):,} tokens."
)

Fetching metadata for 56,599 tokens in 566 batches...
Processed 2,000/56,599
Processed 4,000/56,599
Processed 6,000/56,599
Processed 8,000/56,599
Processed 10,000/56,599
Processed 12,000/56,599
Processed 14,000/56,599
Processed 16,000/56,599
Processed 18,000/56,599
Processed 20,000/56,599
Processed 22,000/56,599
Processed 24,000/56,599
Processed 26,000/56,599
Processed 28,000/56,599
Processed 30,000/56,599
Processed 32,000/56,599
Processed 34,000/56,599
Processed 36,000/56,599
Processed 38,000/56,599
Processed 40,000/56,599
Retry 1/6 (100 tokens): Web3RPCError
Processed 42,000/56,599
Processed 44,000/56,599
Processed 46,000/56,599
Processed 48,000/56,599
Processed 50,000/56,599
Processed 52,000/56,599
Processed 54,000/56,599
Processed 56,000/56,599

Done!
shape: (5, 3)
┌─────────────────────────────────┬──────────┬─────────┐
│ token                           ┆ decimals ┆ symbol  │
│ ---                             ┆ ---      ┆ ---     │
│ str                             ┆ i64      ┆ st

---

## Daily Prices for ETH

here we fetch daily ethereum prices directly from binance. this data serves as a reliable baseline to calculate the US dollar value of swaps that route through wrapped ethereum.

*NOTE:* you need a VPN connection for this part if you're runing it from Iran

In [22]:
print("Fetching daily ETH prices from Binance...")

url = "https://api.binance.com/api/v3/klines"
# Timeframe: Jan 1, 2025 to July 31, 2026
params = {
    "symbol": "ETHUSDT",
    "interval": "1d",
    "startTime": int(datetime(2025, 1, 1, tzinfo=timezone.utc).timestamp() * 1000),
    "endTime": int(datetime(2026, 7, 31, tzinfo=timezone.utc).timestamp() * 1000),
    "limit": 1000
}

response = requests.get(url, params=params)
if response.status_code != 200:
    raise RuntimeError(f"Binance API Error: {response.text}")

data = []
# Binance returns arrays: [Open time, Open, High, Low, Close, Volume, Close time, ...]
for row in response.json():
    dt = datetime.fromtimestamp(row[0] / 1000, tz=timezone.utc)
    data.append({
        "date": dt.date(), 
        "eth_price_usd": float(row[4]) # Using the daily Close price
    })

df_price = pl.DataFrame(data)
df_price.write_parquet("weth_daily_prices.parquet")

print("Done! Saved to weth_daily_prices.parquet")
print(f"Total days fetched: {df_price.height}")
print(df_price.head())

Fetching daily ETH prices from Binance...
Done! Saved to weth_daily_prices.parquet
Total days fetched: 577
shape: (5, 2)
┌────────────┬───────────────┐
│ date       ┆ eth_price_usd │
│ ---        ┆ ---           │
│ date       ┆ f64           │
╞════════════╪═══════════════╡
│ 2025-01-01 ┆ 3360.38       │
│ 2025-01-02 ┆ 3455.67       │
│ 2025-01-03 ┆ 3609.01       │
│ 2025-01-04 ┆ 3656.88       │
│ 2025-01-05 ┆ 3635.99       │
└────────────┴───────────────┘


---

## Binance 1-Minute Prices for ETH-USDT
this cell downloads extremely granular one minute candlestick data from the binance vision data archives. it fetches monthly zip files, unzips them in memory, and normalizes the timestamps to ensure precise pricing for high frequency swaps.

In [26]:
# configuration variables
BASE_URL = "https://data.binance.vision/data/spot/monthly/klines"
SYMBOL = "ETHUSDT"
INTERVAL = "1m"

START_DATE = datetime(2024, 12, 28)
END_DATE = datetime(2026, 7, 1)

OUTPUT_PATH = "binance_weth_1m_prices.parquet"

MAX_RETRIES = 5
RETRY_BACKOFF = 2.0          # seconds, doubled each attempt
US_THRESHOLD = 1_000_000_000_000_000   # above this, epochs are microseconds

BINANCE_COLUMNS = [
    "open_time", "open", "high", "low", "close", "volume",
    "close_time", "quote_volume", "count",
    "taker_buy_volume", "taker_buy_quote_volume", "ignore",
]


# helper functions
def epoch_expr(col: str, alias: str) -> pl.Expr:
    """Binance switched open_time from ms to us in 2025; pick the unit by magnitude."""
    # detects if timestamp is milliseconds or microseconds based on magnitude
    ts = pl.col(col).cast(pl.Int64)
    return (
        pl.when(ts > US_THRESHOLD)
        .then(pl.from_epoch(ts, time_unit="us"))
        .otherwise(pl.from_epoch(ts, time_unit="ms"))
        .alias(alias)
    )


def download_bytes(url: str, month_str: str) -> bytes | None:
    """Fetch a URL with retries. Returns None on a clean 404."""
    last_err: Exception | None = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with urllib.request.urlopen(url, timeout=120) as resp:
                expected = resp.headers.get("Content-Length")
                payload = resp.read()

            if expected is not None and len(payload) != int(expected):
                raise IOError(
                    f"short read: got {len(payload):,} of {int(expected):,} bytes"
                )

            # Reject a corrupt archive here so a retry can still help.
            with zipfile.ZipFile(io.BytesIO(payload)) as zf:
                if zf.testzip() is not None:
                    raise zipfile.BadZipFile("CRC mismatch in archive")

            return payload

        except urllib.error.HTTPError as exc:
            if exc.code == 404:
                print(f"{month_str}: not published yet (404), skipping", flush=True)
                return None
            last_err = exc
        except (
            urllib.error.URLError,
            http.client.IncompleteRead,
            zipfile.BadZipFile,
            zlib.error,
            IOError,
        ) as exc:
            last_err = exc

        if attempt < MAX_RETRIES:
            delay = RETRY_BACKOFF * (2 ** (attempt - 1))
            print(
                f"{month_str}: attempt {attempt} failed ({last_err}), "
                f"retrying in {delay:.0f}s",
                flush=True,
            )
            time.sleep(delay)

    raise RuntimeError(f"{month_str}: failed after {MAX_RETRIES} attempts") from last_err


def fetch_month(month: datetime) -> pl.DataFrame | None:
    month_str = month.strftime("%Y-%m")
    url = f"{BASE_URL}/{SYMBOL}/{INTERVAL}/{SYMBOL}-{INTERVAL}-{month_str}.zip"

    payload = download_bytes(url, month_str)
    if payload is None:
        return None

    with zipfile.ZipFile(io.BytesIO(payload)) as zf:
        with zf.open(zf.namelist()[0]) as f:
            raw = pl.read_csv(
                f,
                has_header=False,
                new_columns=BINANCE_COLUMNS,
                infer_schema_length=0,
            )

    # Drop any header row Binance ships in some archives.
    raw = raw.filter(pl.col("open_time").str.contains(r"^\d+$"))

    df = raw.select(
        epoch_expr("open_time", "price_minute").dt.truncate("1m"),
        pl.col("close").cast(pl.Float64).alias("eth_price_usd"),
    )

    print(f"{month_str}: {df.height:,} rows", flush=True)
    return df


# execution loop
month = datetime(START_DATE.year, START_DATE.month, 1)
last = datetime(END_DATE.year, END_DATE.month, 1)

frames: list[pl.DataFrame] = []
while month <= last:
    print(f"--> starting {month:%Y-%m}", flush=True)
    df = fetch_month(month)
    if df is not None:
        frames.append(df)
    month += relativedelta(months=1)

if not frames:
    raise RuntimeError("no data downloaded")

prices = (
    pl.concat(frames)
    .filter(
        pl.col("price_minute").is_between(START_DATE, END_DATE, closed="both")
    )
    .unique(subset="price_minute", keep="last")
    .sort("price_minute")
)

prices.write_parquet(OUTPUT_PATH)
print(f"wrote {prices.height:,} rows to {OUTPUT_PATH}", flush=True)
print(prices.head(), flush=True)


--> starting 2024-12
2024-12: 44,640 rows
--> starting 2025-01
2025-01: 44,640 rows
--> starting 2025-02
2025-02: 40,320 rows
--> starting 2025-03
2025-03: 44,640 rows
--> starting 2025-04
2025-04: 43,200 rows
--> starting 2025-05
2025-05: 44,640 rows
--> starting 2025-06
2025-06: 43,200 rows
--> starting 2025-07
2025-07: 44,640 rows
--> starting 2025-08
2025-08: 44,640 rows
--> starting 2025-09
2025-09: 43,200 rows
--> starting 2025-10
2025-10: 44,640 rows
--> starting 2025-11
2025-11: 43,200 rows
--> starting 2025-12
2025-12: 44,640 rows
--> starting 2026-01
2026-01: 44,640 rows
--> starting 2026-02
2026-02: 40,320 rows
--> starting 2026-03
2026-03: 44,640 rows
--> starting 2026-04
2026-04: 43,200 rows
--> starting 2026-05
2026-05: 44,640 rows
--> starting 2026-06
2026-06: 43,200 rows
--> starting 2026-07
2026-07: 44,640 rows
wrote 792,001 rows to binance_weth_1m_prices.parquet
shape: (5, 2)
┌─────────────────────┬───────────────┐
│ price_minute        ┆ eth_price_usd │
│ ---        

---

## Applies decimals and daily prices, creates swaps_usd folder

this cell merges our cleaned swap data with the pool registry and token metadata. it uses the token decimals to convert raw blockchain integers into human readable amounts, and then estimates the US dollar volume of each trade by checking against popular stablecoins or multiplying by the ethereum daily price.

In [29]:
OUTDIR = Path("swaps_usd")
OUTDIR.mkdir(exist_ok=True)

# Mainnet addresses (must be lowercase)
STABLES = [
    "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48", # USDC
    "0xdac17f958d2ee523a2206206994597c13d831ec7", # USDT
    "0x6b175474e89094c44da98b954eedeac495271d0f"  # DAI
]
WETH = "0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2"

# Load Lookups
print("Loading lookups...")
pools = pl.read_parquet("uniswap_v3_pools_clean.parquet").select([
    pl.col("pool").str.to_lowercase().alias("pool_address"), 
    pl.col("token0").str.to_lowercase(), 
    pl.col("token1").str.to_lowercase()
])

tokens = pl.read_parquet("token_metadata.parquet").select([
    pl.col("token").str.to_lowercase(), 
    "decimals", 
    "symbol"
])

prices = pl.read_parquet("weth_daily_prices.parquet")

# Process Batches
batch_files = sorted(Path("clean").glob("*.parquet"))
print(f"Found {len(batch_files)} clean batches to process.")

for f in batch_files:
    out_path = OUTDIR / f.name
    # if out_path.exists():
    #     print(f"Skip {f.name}")
    #     continue
        
    df = pl.read_parquet(f)
    df = df.with_columns(pl.col("block_time").cast(pl.Date).alias("date"))
    
    # merge metadata onto the transaction logs
    df = df.join(pools, on="pool_address", how="left")
    df = df.join(tokens, left_on="token0", right_on="token", how="left").rename({"decimals": "decimals0", "symbol": "symbol0"})
    df = df.join(tokens, left_on="token1", right_on="token", how="left").rename({"decimals": "decimals1", "symbol": "symbol1"})
    df = df.join(prices, on="date", how="left")
    
    # adjust amounts according to token decimals utilizing float exponentiation
    # THE FIX: pl.lit(10.0) forces Float64 exponentiation, preventing the Int32 overflow!
    df = df.with_columns([
        (pl.col("amount0_f64") / (pl.lit(10.0).pow(pl.col("decimals0")))).alias("amount0_adj"),
        (pl.col("amount1_f64") / (pl.lit(10.0).pow(pl.col("decimals1")))).alias("amount1_adj")
    ])
    
    # establish unified trading volume metric
    df = df.with_columns(
        pl.when(pl.col("token0").is_in(STABLES))
          .then(pl.col("amount0_adj").abs())
          .when(pl.col("token1").is_in(STABLES))
          .then(pl.col("amount1_adj").abs())
          .when(pl.col("token0") == WETH)
          .then(pl.col("amount0_adj").abs() * pl.col("eth_price_usd"))
          .when(pl.col("token1") == WETH)
          .then(pl.col("amount1_adj").abs() * pl.col("eth_price_usd"))
          .otherwise(None)
          .alias("amount_usd")
    )
    
    df = df.drop(["date", "eth_price_usd"])
    df.write_parquet(out_path, compression="zstd")
    
    batch_usd = df["amount_usd"].sum()
    print(f"{f.name}: {len(df):,} rows | Batch Volume: ${batch_usd:,.2f}")

print("\n Dataset B is built correctly!")

Loading lookups...
Found 24 clean batches to process.
uniswap_v3_swaps_2025-01-01_to_2025-04-01__blocks_21500000_21700000.parquet: 2,586,295 rows | Batch Volume: $37,574,769,552.74
uniswap_v3_swaps_2025-01-01_to_2025-04-01__blocks_21700000_21900000.parquet: 3,255,821 rows | Batch Volume: $36,719,454,276.32
uniswap_v3_swaps_2025-01-01_to_2025-04-01__blocks_21900000_22100000.parquet: 3,438,146 rows | Batch Volume: $34,378,903,440.69
uniswap_v3_swaps_2025-01-01_to_2025-04-01__blocks_22100000_22148000.parquet: 662,532 rows | Batch Volume: $4,535,950,305.29
uniswap_v3_swaps_2025-04-01_to_2025-07-01__blocks_22148000_22348000.parquet: 3,245,002 rows | Batch Volume: $20,269,199,856.87
uniswap_v3_swaps_2025-04-01_to_2025-07-01__blocks_22348000_22548000.parquet: 3,065,328 rows | Batch Volume: $20,997,497,306.37
uniswap_v3_swaps_2025-04-01_to_2025-07-01__blocks_22548000_22748000.parquet: 2,496,671 rows | Batch Volume: $17,890,852,911.47
uniswap_v3_swaps_2025-04-01_to_2025-07-01__blocks_22748000_2

---
**Results and Data Integrity:**

This execution yielded **Dataset C** (Pool and Token Metadata). More importantly, it enriched the foundational ledger from Phase I, appending exact USD nominal values to all swap legs. The integration of decimal normalization and Oracle pricing ensures that the subsequent econometric modeling of slippage and sandwich attacks is grounded in precise economic reality, avoiding the severe multi-collinearity issues common in amateur DeFi research.